In [ ]:
%matplotlib inline
import pandas as pd
import scipy
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import sktime
import sklearn
from scipy.io import loadmat
from sktime.transformations.panel.rocket import Rocket
# import timesynth as ts
import random
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

In [ ]:
auc_random_25 = []
auc_random_50 = []
auc_random_75 = []

accuracy_random_25 = []
accuracy_random_50 = []
accuracy_random_75 = []

n_random_runs = 100
# random metrics
for i in range(10):

    X_train = np.load(
        "C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy"
    )
    X_test = np.load(
        "C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy"
    )
    y_train = np.load(
        "C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy"
    )
    y_test = np.load(
        "C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy"
    )


    # Train model
    trf = Rocket(num_kernels=1000, random_state=1234)

    rocket_features = trf.fit_transform(X_train)

    scaler = StandardScaler()
    rocket_features = scaler.fit_transform(rocket_features)

    model = LogisticRegression(max_iter=1000)
    model.fit(rocket_features, y_train)


    n_points = X_test.shape[2]


    auc_runs_25 = []
    auc_runs_50 = []
    auc_runs_75 = []

    acc_runs_25 = []
    acc_runs_50 = []
    acc_runs_75 = []


    for run in range(n_random_runs):

        X_test_random25 = X_test.copy()
        X_test_random50 = X_test.copy()
        X_test_random75 = X_test.copy()


        # store the SAME random ranking for each sample
        for sample in range(X_test.shape[0]):

            random_order = np.random.permutation(n_points)


            idx25 = random_order[:int(0.05*n_points)]
            idx50 = random_order[:int(0.10*n_points)]
            idx75 = random_order[:int(0.15*n_points)]


            mu = np.mean(X_test[sample])


            X_test_random25[
                sample,0,idx25
            ] = mu

            X_test_random50[
                sample,0,idx50
            ] = mu

            X_test_random75[
                sample,0,idx75
            ] = mu



        # ---- 25% ----
        feat25 = scaler.transform(
            trf.transform(X_test_random25)
        )

        prob25 = model.predict_proba(feat25)[:,1]
        pred25 = model.predict(feat25)

        auc_runs_25.append(
            roc_auc_score(y_test, prob25)
        )

        acc_runs_25.append(
            accuracy_score(y_test, pred25)
        )


        # ---- 50% ----
        feat50 = scaler.transform(
            trf.transform(X_test_random50)
        )

        prob50 = model.predict_proba(feat50)[:,1]
        pred50 = model.predict(feat50)

        auc_runs_50.append(
            roc_auc_score(y_test, prob50)
        )

        acc_runs_50.append(
            accuracy_score(y_test, pred50)
        )


        # ---- 75% ----
        feat75 = scaler.transform(
            trf.transform(X_test_random75)
        )

        prob75 = model.predict_proba(feat75)[:,1]
        pred75 = model.predict(feat75)

        auc_runs_75.append(
            roc_auc_score(y_test, prob75)
        )

        acc_runs_75.append(
            accuracy_score(y_test, pred75)
        )


    # average random runs for this split
    auc_random_25.append(np.mean(auc_runs_25))
    auc_random_50.append(np.mean(auc_runs_50))
    auc_random_75.append(np.mean(auc_runs_75))

    accuracy_random_25.append(np.mean(acc_runs_25))
    accuracy_random_50.append(np.mean(acc_runs_50))
    accuracy_random_75.append(np.mean(acc_runs_75))


    print("Split", i)
    print("Random 25 AUC:", np.mean(auc_runs_25))
    print("Random 50 AUC:", np.mean(auc_runs_50))
    print("Random 75 AUC:", np.mean(auc_runs_75))

In [ ]:
np.mean(auc_random_75)

In [ ]:
pd.DataFrame(auc_random_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_random/auc_top5rem.csv', index=False)
pd.DataFrame(auc_random_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_random/auc_top15rem.csv', index=False)
pd.DataFrame(auc_random_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_random/auc_top10rem.csv', index=False)

pd.DataFrame(accuracy_random_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_random/accuracy_top5rem.csv', index=False)
pd.DataFrame(accuracy_random_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_random/accuracy_top10rem.csv', index=False)
pd.DataFrame(accuracy_random_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_random/accuracy_top15rem.csv', index=False)

In [ ]:
auc_75 = []
accuracy_75 = []
auc_50 = []
accuracy_50 = []
auc_25 = []
accuracy_25 = []
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np
import pandas as pd
import shap
 
auc_75 = []
accuracy_75 = []
auc_50 = []
accuracy_50 = []
auc_25 = []
accuracy_25 = []
for i in range(5,10):
    rocket_features = pd.read_csv('C:/Users/satavans/Downloads/alcohol_application/train_files/driving_split' + str(i) + '.csv')
    X_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/X_train" + str(i) + ".npy")
    X_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/X_test" + str(i) + ".npy")
    y_train = np.load("C:/Users/satavans/Downloads/alcohol_application/train_raw/y_train" + str(i) + ".npy")
    y_test = np.load("C:/Users/satavans/Downloads/alcohol_application/test_raw/y_test" + str(i) + ".npy")
    X_test2 = X_test.copy()
 
    # fit the rocket transformer on sober drive data
    trf = Rocket(num_kernels=1000, random_state = 1234)
    rocket_features = trf.fit_transform(X_train)
    model = LogisticRegression(max_iter = 1000)
    scaler = StandardScaler()
    rocket_features = scaler.fit_transform(rocket_features)
    model.fit(rocket_features, y_train)
 
    X_test_rocket = trf.transform(X_test)
    X_test_rocket = scaler.transform(X_test_rocket)
    y_pred = model.predict(X_test_rocket)
    print("Original Accuracy:", accuracy_score(y_pred, y_test))
    y_prob = model.predict_proba(X_test_rocket)[:,1]
 
    print("Original AUC: ", roc_auc_score(y_test, y_prob))
 
     # 3. Define the wrapper function
    # 2. Define the wrapper function
    def rocket_predict(X_2d):
        # SHAP passes (samples, time_steps). Inject the middle '1' dimension back for 3D ROCKET
        X_3d = X_2d.reshape(-1, 1, 3600)
        X_tf = trf.transform(X_3d)
        X_tf = scaler.transform(X_tf)
        return model.decision_function(X_tf)
 
    # 4. Initialize KernelExplainer with a background dataset (often a KMeans summary of train data)
    # Note: For time series, explaining the raw points requires careful consideration of feature independence
    X_train_2d = X_train.squeeze(axis=1)
    background_data = shap.sample(X_train_2d, 10)
    explainer = shap.KernelExplainer(rocket_predict, background_data)
 
    # 4. Prepare 2D background data and initialize LimeTabularExplainer
    # Squeeze 3D train data to 2D (samples, time_steps)
    X_test_2d = X_test.squeeze(axis=1)
 
    TP = FP = FN = 0
    total_time_points = X_test.shape[2]
    all_lime = []
    for sample in range(X_test.shape[0]):
        # 2. Extract continuous SHAP values for this sample and take absolute magnitudes
        # 2. Extract continuous SHAP values for this sample and take absolute magnitudes
        all_lime.append(
            np.abs(explainer.shap_values(X_test_2d[sample], nsamples = 100, l1_reg = "num_features(100)"))
            )
 
        sample_shap = all_lime[sample]
 
        ## get top k (# of spikes) points
        n_points = len(sample_shap)
        n_del = int(0.05 * n_points)
        # len(changed_set)
        delt = np.argsort(sample_shap)[-n_del:]
        mask = np.zeros(len(sample_shap), dtype=bool)
        mask[delt] = True
 
        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])
 
        X_test2[sample,0,mask] = mu
 
    X_test_rocket2 = trf.transform(X_test2)
    X_test_rocket2 = scaler.transform(X_test_rocket2)
    y_pred2 = model.predict(X_test_rocket2)
    print("Top 5 Accuracy:", accuracy_score(y_pred2, y_test))
    accuracy_75.append(accuracy_score(y_pred2, y_test))
    y_prob2 = model.predict_proba(X_test_rocket2)[:,1]
 
    print("Top 5 removed AUC: ", roc_auc_score(y_test, y_prob2))
    auc_75.append(roc_auc_score(y_test, y_prob2))
 
    total_time_points = X_test.shape[2]
 
    X_test2 = X_test.copy()
    for sample in range(X_test.shape[0]):
 
        sample_shap = all_lime[sample]
        ## get top k (# of spikes) points
        n_points = len(sample_shap)
        n_del = int(0.1 * n_points)
        # len(changed_set)
        delt = np.argsort(sample_shap)[-n_del:]
        mask = np.zeros(len(sample_shap), dtype=bool)
        mask[delt] = True
       
        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])
 
        X_test2[sample,0,mask] = mu
 
    X_test_rocket2 = trf.transform(X_test2)
    X_test_rocket2 = scaler.transform(X_test_rocket2)
    y_pred2 = model.predict(X_test_rocket2)
    print("Top 10 Accuracy:", accuracy_score(y_pred2, y_test))
    accuracy_50.append(accuracy_score(y_pred2, y_test))
    y_prob2 = model.predict_proba(X_test_rocket2)[:,1]
 
    print("Top 10 removed AUC: ", roc_auc_score(y_test, y_prob2))
    auc_50.append(roc_auc_score(y_test, y_prob2))
 
    X_test2 = X_test.copy()
    for sample in range(X_test.shape[0]):
        sample_shap = all_lime[sample]
        ## get top k (# of spikes) points
        n_points = len(sample_shap)
        n_del = int(0.15 * n_points)
        # len(changed_set)
        delt = np.argsort(sample_shap)[-n_del:]
        mask = np.zeros(len(sample_shap), dtype=bool)
        mask[delt] = True
       
        mu = np.mean(X_test[sample])
        sigma = np.std(X_test[sample])
 
        X_test2[sample,0,mask] = mu
 
    X_test_rocket2 = trf.transform(X_test2)
    X_test_rocket2 = scaler.transform(X_test_rocket2)
    y_pred2 = model.predict(X_test_rocket2)
    print("Top 15 Accuracy:", accuracy_score(y_pred2, y_test))
    accuracy_25.append(accuracy_score(y_pred2, y_test))
    y_prob2 = model.predict_proba(X_test_rocket2)[:,1]
 
    print("Top 15 removed AUC: ", roc_auc_score(y_test, y_prob2))
    auc_25.append(roc_auc_score(y_test, y_prob2))
 

In [ ]:
pd.DataFrame(auc_random_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_shap/auc_top5rem1.csv', index=False)
pd.DataFrame(auc_random_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_shap/auc_top15rem1.csv', index=False)
pd.DataFrame(auc_random_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_shap/auc_top10rem1.csv', index=False)

pd.DataFrame(accuracy_random_25).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_shap/accuracy_top5rem1.csv', index=False)
pd.DataFrame(accuracy_random_50).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_shap/accuracy_top10rem1.csv', index=False)
pd.DataFrame(accuracy_random_75).to_csv('C:/Users/satavans/Downloads/alcohol_application/deletion_results_shap/accuracy_top15rem1.csv', index=False)